# 💰 LoanGenie — Advanced Agentic AI (Part 2)
### Multi-Agent · Streaming · Guardrails & PII · Observability

---

This notebook is the **advanced continuation** of the LoanGenie project. It assumes you have
already completed the console setup and the core modules (1–10) in Part 1:

| Resource | Expected |
|----------|----------|
| Bedrock model access | Amazon Nova Pro enabled |
| DynamoDB table | `LoanApplicants` |
| AgentCore memory | `loan_applicants` namespace |
| Knowledge Base | `LoanProductsKB` (synced) |
| SageMaker role | 8 policies attached |

**What you'll add here — production-grade capabilities:**

| Module | Capability | Why it matters |
|--------|-----------|----------------|
| A | Multi-agent orchestration | Split eligibility, advisory & docs into specialist agents |
| B | Streaming + async tools | Real-time token streaming, parallel tool calls |
| C | Guardrails + PII redaction | Mask Aadhaar/PAN, block toxic content — compliance |
| D | Observability | CloudWatch metrics, tracing, per-query cost tracking |

> **Region:** `us-east-1` · **Model:** Amazon Nova Pro

---
## 🚨 IMPORTANT — Read Before Running

This notebook has **three things you MUST customize** for your account. Look for the 🔧 **REPLACE THIS** callouts below:

| # | What to replace | Where |
|---|-----------------|-------|
| 1 | `KB_ID` — your Knowledge Base ID | Config cell (Cell 4) |
| 2 | `AGENTCORE_MEM_ID` — your AgentCore Memory ID | Config cell (Cell 4) |
| 3 | `S3_BUCKET` — your Lambda deployment bucket | Cell E-3 (packaging) |

**Everything else is created dynamically** using your AWS Account ID, so no other manual changes are needed.

⚠️ **Common trap:** If you skip Cell 2 (dependency fix) and see a `500 Internal Server Error` from the API later — the root cause is almost always a **pip dependency conflict** from a newer SageMaker base image. Run Cell 2 first and **always restart the kernel after.**

In [ ]:
# 🔧 FIX FOR DEPENDENCY CONFLICTS — Run this FIRST
#
# Newer SageMaker base images ship with botocore/attrs/packaging versions that
# clash with what sagemaker 2.257.3 and awscli 1.45.42 expect. This forces
# compatible versions so the Lambda package built later doesn't hit
# "pip's dependency resolver does not currently take into account..." errors.

!pip install \
    "botocore==1.43.42" \
    "attrs>=24,<26" \
    "packaging>=23.0,<25" \
    --quiet

print('✅ Dependency versions pinned')
print('⚠️  IMPORTANT: Restart the kernel now (Kernel → Restart Kernel) before running the next cell')

### 📖 What this cell does

**The problem:** When SageMaker Studio creates a new JupyterLab space, it pulls the **latest** base image — which includes newer versions of `botocore`, `attrs`, and `packaging` than the AWS CLI and SageMaker SDK expect. When you later package this code into a Lambda zip, those version conflicts cause the Lambda to fail at runtime with a **500 error** from API Gateway.

**The fix:** Force-install the older, compatible versions:
- `botocore==1.43.42` (matches what `awscli 1.45.42` requires)
- `attrs>=24,<26` (matches what `sagemaker 2.257.3` requires)
- `packaging>=23.0,<25` (matches what `sagemaker 2.257.3` requires)

**Why `--quiet`?** Suppresses hundreds of lines of pip output so the notebook stays readable.

**Why is this safe for older environments?** If your environment already has compatible versions, pip sees the requirements are satisfied and skips the install — no downgrade, no disruption.

> 🔄 **Restart the kernel after this cell runs** (Kernel menu → Restart Kernel). Python has already imported the old versions into memory; a restart makes it pick up the new ones.

In [ ]:
# Install the Strands Agents framework (the agentic AI library we'll use)
!pip install strands-agents strands-agents-tools boto3 -q
print('✅ Strands + boto3 installed')

### 📖 What this cell does

Installs three Python packages needed for the rest of the notebook:

| Package | Purpose |
|---------|---------|
| `strands-agents` | Core agentic framework — lets us define `Agent`, `@tool`, and orchestrate LLM calls with tool use |
| `strands-agents-tools` | Pre-built tool integrations (retrieval, streaming, etc.) |
| `boto3` | AWS SDK for Python — used to call Bedrock, DynamoDB, Lambda, API Gateway, S3, CloudWatch |

The `-q` flag keeps output minimal. You should see just `✅ Strands + boto3 installed` if everything worked.

---
## 🔧 REPLACE THIS — Update Your Resource IDs

The next cell contains three values you **must replace** with IDs from your own AWS account:

| Variable | Where to find it |
|----------|------------------|
| `KB_ID` | Bedrock console → Knowledge Bases → `LoanProductsKB` → copy **Knowledge base ID** |
| `AGENTCORE_MEM_ID` | Bedrock AgentCore console → Memory → your memory → copy **Memory ID** |
| `APPLICANT_TABLE` | Leave as `LoanApplicants` (unless you named your DynamoDB table differently) |

⚠️ Don't skip this — if the KB_ID is wrong, the `search_loan_products` tool will fail silently returning *No relevant information found*.

In [ ]:
# [CONFIG] — resource IDs + ALL imports used anywhere in this notebook
# (centralized here so re-running any single cell later never hits a missing-import NameError)
import boto3, json, os, time, re, uuid, asyncio, subprocess, zipfile, shutil
import urllib.request, urllib.error
from datetime import datetime, timezone

REGION     = 'us-east-1'
MODEL_ID   = 'us.amazon.nova-pro-v1:0'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

# 🔧 REPLACE THESE THREE VALUES ↓↓↓
KB_ID            = 'HGLSKTH8OX'                     # ← Paste your LoanProductsKB ID
AGENTCORE_MEM_ID = 'application_loan-Ro4cqL8XUg'    # ← Paste your loan_applicants Memory ID
APPLICANT_TABLE  = 'LoanApplicants'                 # ← Change only if your DynamoDB table name differs

print(f'Account : {ACCOUNT_ID}')
print(f'KB_ID   : {KB_ID}')
print('✅ Config + imports loaded')

### 📖 What this cell does

**Centralized configuration** — every downstream cell reads its resource IDs from here, so if you re-run any single cell later you won't hit `NameError: KB_ID is not defined`.

**Key variables:**
- `REGION` — AWS region. Bedrock Nova Pro is available in `us-east-1`; don't change this unless you know the model is available in your region.
- `MODEL_ID` — Amazon Nova Pro inference profile ID. The `us.` prefix means it's a **cross-region inference profile** (Nova Pro's default).
- `ACCOUNT_ID` — pulled automatically from your STS caller identity, so you never have to hardcode it.
- `KB_ID`, `AGENTCORE_MEM_ID`, `APPLICANT_TABLE` — the three resources you created in the console during Part 1.

**Why all imports are here:** If any cell later needs `uuid` or `re` or `subprocess`, it's already imported. This avoids the classic "I re-ran cell 15 and now it says NameError" trap.

In [ ]:
# Markdown rendering helper — use show() instead of print() for formatted output
from IPython.display import Markdown, display

def show(response, title=None):
    """Render an agent response as clean formatted markdown in the notebook."""
    text = str(response)
    if title:
        text = f'### {title}\n\n' + text
    display(Markdown(text))

print('✅ show() markdown renderer ready — use it instead of print() for agent output')

### 📖 What this cell does

Defines a helper function `show()` that renders text as **formatted markdown** in the notebook output cell — instead of the raw `**bold**` and `## headers` you'd see with `print()`.

**Why we need this:** Our agents are instructed to reply in markdown (headers, bold, bullets). Using `print()` would show the raw `##` and `**` characters. Using `show()` renders them as actual formatted headers and bold text — much nicer to read.

**Usage pattern throughout the notebook:**
```python
response = agent("Am I eligible for a loan?")
show(response)  # renders formatted, not print(response)
```

---
## Shared Financial Tools
These are reused by every specialist agent below.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def check_eligibility(monthly_income: float, credit_score: float, loan_amount: float) -> str:
    """Check loan eligibility from income, credit score, and requested amount."""
    max_loan = monthly_income * 60
    reasons, eligible = [], True
    if credit_score < 650: eligible=False; reasons.append('Credit score below 650')
    if loan_amount > max_loan: eligible=False; reasons.append('Exceeds max eligible')
    if monthly_income < 25000: eligible=False; reasons.append('Income below 25000')
    band = '10.5%-12.5%' if credit_score >= 750 else '12.5%-15%'
    return json.dumps({'eligible':eligible,'max_eligible_amount':int(max_loan),
                       'reasons':reasons or ['Meets all criteria'],'interest_rate_band':band})

@tool
def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    """Calculate monthly EMI, total payable, and total interest."""
    r = annual_rate/12/100
    emi = principal/tenure_months if r==0 else principal*r*(1+r)**tenure_months/((1+r)**tenure_months-1)
    return json.dumps({'monthly_emi':round(emi,2),'total_payable':round(emi*tenure_months,2),
                       'total_interest':round(emi*tenure_months-principal,2)})

@tool
def search_loan_products(query: str) -> str:
    """Search loan product terms, rates, documents, and FAQs from the knowledge base."""
    if not KB_ID:
        return 'Knowledge base not configured.'
    client = boto3.client('bedrock-agent-runtime', region_name=REGION)
    resp = client.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'managedSearchConfiguration': {}   # ← correct for managed KB
        }
    )
    results = resp.get('retrievalResults', [])
    if not results:
        return 'No relevant information found.'
    return '\n\n'.join(
        f'[{r.get("location",{}).get("s3Location",{}).get("uri","doc").split("/")[-1]}] {r["content"]["text"]}'
        for r in results
    )

### 📖 What this cell does

Defines the **three core tools** every specialist agent will use. Each is wrapped with `@tool` — a Strands decorator that turns a Python function into a callable tool the LLM can decide to invoke.

**Tool 1: `check_eligibility`** — Pure Python business logic. Applies the four eligibility rules from the system prompt (income ≥ 25k, credit score ≥ 650, loan ≤ 60× income, rate band by credit score) and returns a JSON verdict. **No LLM involved** — deterministic, fast, cheap.

**Tool 2: `calculate_emi`** — Standard EMI formula: `EMI = P × r × (1+r)^n / ((1+r)^n − 1)`. Returns monthly payment, total payable, and total interest.

**Tool 3: `search_loan_products`** — Calls the Bedrock **Knowledge Base retrieve API** (RAG). Sends the query to your `LoanProductsKB`, gets back the top matching chunks from your S3-hosted documents, and formats them with source filenames.

**Why tools instead of prompt-only?** The LLM is unreliable at math and doesn't know your document contents. Tools let the LLM decide *when* to call code that produces exact, grounded answers — the ReAct pattern (Thought → Action → Observation → Answer).

---
# MODULE A — Multi-Agent Orchestration
### A supervisor routes each query to the right specialist

In production, a single agent juggling every task becomes hard to tune and debug. The
supervisor pattern splits responsibility:

```
                    ┌──────────────────┐
   User query  ───▶ │  Supervisor Agent │  (routes by intent)
                    └────────┬─────────┘
              ┌──────────────┼──────────────┐
              ▼              ▼              ▼
      ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
      │ Eligibility │ │  Advisory   │ │ Documents   │
      │ Specialist  │ │ Specialist  │ │ Specialist  │
      └─────────────┘ └─────────────┘ └─────────────┘
```

Each specialist has a focused prompt and only the tools it needs — easier to test,
cheaper to run, and safer for compliance.

In [ ]:
# Define the three specialist agents as callable tools for the supervisor

@tool
def eligibility_specialist(query: str) -> str:
    """Handle loan eligibility questions."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are an eligibility specialist for an Indian lending company.
Use check_eligibility to assess applications.
Format in markdown with **bold** for key figures.
No reasoning steps in output. Final answer only.''',
        tools=[check_eligibility]
    )
    return str(agent(query))

@tool
def advisory_specialist(query: str) -> str:
    """Handle EMI calculations and loan advice."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are a loan advisory specialist for an Indian lending company.
Use calculate_emi for repayment figures.
Use search_loan_products for rates and terms.
Format in markdown with **bold** for key figures.
No reasoning steps in output. Final answer only.''',
        tools=[calculate_emi, search_loan_products]
    )
    return str(agent(query))

@tool
def documents_specialist(query: str) -> str:
    """Handle document and process questions."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are a documentation specialist for an Indian lending company.
ALWAYS call search_loan_products first before answering.
Only state documents returned by the knowledge base — PAN, Aadhaar, ITR, salary slips etc.
Never use generic documents like driver license or passport.
Format in markdown with bullet points.
No reasoning steps in output. Final answer only.''',
        tools=[search_loan_products]
    )
    return str(agent(query))

print('✅ Three specialist agents defined')

### 📖 What this cell does

Creates three **specialist sub-agents**, each wrapped as a `@tool` so the supervisor can call them like any other function.

**The pattern:** Each specialist is itself an `Agent` with:
1. A **focused system prompt** (only its job)
2. A **narrow tool set** (only what it needs)
3. **Strict formatting rules** ("No reasoning steps in output. Final answer only.")

| Specialist | Owns | Tools |
|------------|------|-------|
| `eligibility_specialist` | Income/credit/loan verdicts | `check_eligibility` |
| `advisory_specialist` | EMI math, product rates | `calculate_emi`, `search_loan_products` |
| `documents_specialist` | KYC & doc requirements | `search_loan_products` (grounded RAG only) |

**Why this beats one big agent:**
- **Easier to debug** — if EMI is wrong, you know exactly which agent to inspect
- **Cheaper** — specialists use smaller prompts and fewer tool calls per query
- **Safer** — the documents specialist is *forced* to hit the KB first, so it can't hallucinate "you need a passport"
- **Independently tunable** — you can retrain or reprompt one specialist without breaking the others

In [ ]:
# The supervisor agent — routes to specialists
supervisor = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='''You are the LoanGenie supervisor. Route each customer query to the
right specialist tool:
- eligibility_specialist: eligibility, income/credit assessment
- advisory_specialist: EMI calculations, rates, repayment advice
- documents_specialist: required documents, processing times

For multi-part questions, call multiple specialists and combine their answers.
Present a single, coherent response to the customer.

Format your answer in clean markdown:
- Use ## headers for each section (Eligibility, EMI, Documents)
- Use **bold** for key figures like amounts and rates
- Use bullet points for lists
- No internal reasoning or thinking steps in the output''',
    tools=[eligibility_specialist, advisory_specialist, documents_specialist]
)

# Test routing — a multi-part query should hit multiple specialists
response = supervisor(
    'I earn 85000 a month with a credit score of 740. Am I eligible for a 500000 loan, '
    'what would the EMI be over 36 months at 13%, and what documents will I need?'
)
show(response)

### 📖 What this cell does

Creates the **supervisor agent** — the top of the hierarchy. It doesn't answer questions itself; it *routes* them to the right specialist(s).

**How routing works:** The supervisor sees the three specialists as `tools`. When a query comes in, the LLM reads its system prompt (which explains each specialist's role) and picks the right one — or picks several for multi-part questions.

**The test query hits all three specialists:**
1. *"Am I eligible for 500000?"* → `eligibility_specialist`
2. *"What would the EMI be at 13% over 36 months?"* → `advisory_specialist`
3. *"What documents will I need?"* → `documents_specialist`

The supervisor **combines all three answers** into one clean, sectioned response. You'll see the tool calls in the streaming output — `Tool #1: eligibility_specialist`, `Tool #2: advisory_specialist`, `Tool #3: documents_specialist`.

**Watch for:** Sometimes the LLM shows its `<thinking>` in output. That's not a bug in your code — it's Nova Pro's chain-of-thought. In production, you'd strip it with a post-processor.

In [ ]:
# Test individual routing — confirm the supervisor picks the right specialist
tests = [
    'Am I eligible for 400000 with income 70000 and score 690?',   # -> eligibility
    'What is the EMI on 600000 at 12% for 48 months?',              # -> advisory
    'What documents does a salaried applicant need?',               # -> documents
]
for t in tests:
    print(f'\n❓ {t}')
    show(supervisor(t))
    print('-'*60)

### 📖 What this cell does

**Validates the routing logic** with three single-intent queries. Each should hit exactly one specialist:

| Query | Should route to | Why |
|-------|-----------------|-----|
| *"Am I eligible for 400000..."* | `eligibility_specialist` | Contains income + credit + amount |
| *"What is the EMI on 600000..."* | `advisory_specialist` | Pure EMI calculation |
| *"What documents does a salaried applicant need?"* | `documents_specialist` | Pure documentation question |

**Why this matters:** If a single-intent query accidentally hits all three specialists, you're paying 3× the cost per query. This test confirms the supervisor's system prompt is disambiguating correctly.

**Watch for:** In the output, count how many `Tool #N:` lines appear per query. It should be **1 tool call for each single-intent test**.

---
# MODULE B — Streaming Responses + Async Tool Calls
### Stream tokens as they generate; run independent tools in parallel

**Why streaming?** A loan advisory answer can take several seconds. Streaming shows the
customer text as it's generated instead of a frozen spinner — critical for chat UX.

**Why async tools?** When a query needs both an EMI calculation and a product lookup,
running them concurrently cuts latency roughly in half.

In [ ]:
# Streaming responses — token-by-token output using Strands async iterator
import asyncio

async def stream_response(agent, query):
    """Stream the agent's response token by token."""
    print('🤖 ', end='', flush=True)
    async for event in agent.stream_async(query):
        if 'data' in event:
            print(event['data'], end='', flush=True)
    print()

stream_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor. Be concise and helpful.',
    tools=[check_eligibility, calculate_emi, search_loan_products]
)

# Run the streaming demo
await stream_response(stream_agent, 'Explain how personal loan interest rates are decided.')

### 📖 What this cell does

Demonstrates **token-by-token streaming** via Strands' `stream_async()` iterator. Instead of waiting for the complete answer, tokens appear as the model generates them.

**How it works:**
1. `agent.stream_async(query)` returns an **async generator** — a stream of events.
2. We loop with `async for event in ...` and check for events with a `'data'` key (those contain new text).
3. `print(..., end='', flush=True)` writes each token immediately without a newline or buffering delay.

**Why `await` at the top level works:** Jupyter has a built-in async event loop, so you can call `await` directly in a cell (no `asyncio.run()` needed).

**In production:** You'd wire this stream into a WebSocket or Server-Sent Events endpoint, so your web frontend gets tokens as they arrive — giving that "ChatGPT typing" effect users expect from chat interfaces.

In [ ]:
# Direct Bedrock streaming via converse_stream — lower-level control
bedrock = boto3.client('bedrock-runtime', region_name=REGION)

def stream_bedrock(prompt):
    """Stream directly from Bedrock's converse_stream API."""
    resp = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{'role':'user','content':[{'text':prompt}]}],
        inferenceConfig={'maxTokens':300,'temperature':0.3}
    )
    print('🤖 ', end='', flush=True)
    for event in resp['stream']:
        if 'contentBlockDelta' in event:
            delta = event['contentBlockDelta']['delta'].get('text', '')
            print(delta, end='', flush=True)
    print()

stream_bedrock('In one paragraph, explain what a credit score means for loan eligibility.')

### 📖 What this cell does

Same idea as the previous cell, but **one level lower** — calling Bedrock's `converse_stream()` API directly (skipping the Strands abstraction).

**When would you use this?**
- When you don't need tools (just plain LLM completion)
- When you want fine-grained control over token deltas
- When integrating with a system that doesn't play well with Strands

**Key differences vs Strands streaming:**
| | Strands `stream_async` | Bedrock `converse_stream` |
|--|-----------------------|---------------------------|
| Async | Yes (async generator) | No (regular for-loop) |
| Tool use support | Yes | Not shown here |
| Event shape | `event['data']` | `event['contentBlockDelta']['delta']['text']` |
| Best for | Full agents with tools | Pure completion / minimal setups |

**Params to know:**
- `maxTokens: 300` — hard cap on response length
- `temperature: 0.3` — low = deterministic, high = creative. `0.3` is a sensible default for factual assistants.

In [ ]:
# Async parallel tool calls — run independent lookups concurrently
import asyncio, time

async def async_eligibility(income, score, amount):
    return json.loads(check_eligibility(income, score, amount))

async def async_emi(principal, rate, months):
    return json.loads(calculate_emi(principal, rate, months))

async def parallel_loan_analysis(income, score, amount, rate, months):
    """Run eligibility and EMI calculations concurrently."""
    start = time.time()
    eligibility, emi = await asyncio.gather(
        async_eligibility(income, score, amount),
        async_emi(amount, rate, months)
    )
    elapsed = (time.time() - start) * 1000
    return {'eligibility': eligibility, 'emi': emi, 'elapsed_ms': round(elapsed, 1)}

result = await parallel_loan_analysis(85000, 740, 500000, 13, 36)
print(json.dumps(result, indent=2))
print(f'\n✅ Both tools ran concurrently in {result["elapsed_ms"]}ms')

### 📖 What this cell does

Shows how to **run two tools in parallel** using `asyncio.gather()` — cutting total latency in roughly half.

**The pattern:**
```python
result_a, result_b = await asyncio.gather(
    tool_a(...),   # starts immediately
    tool_b(...)    # starts immediately
)                  # waits for BOTH to finish
```

**Why this works here:** Eligibility check and EMI calculation are **independent** — one doesn't need the other's output. When calls are independent, `asyncio.gather` fires them in parallel.

**When NOT to use this:** If tool B needs tool A's result (e.g., "check eligibility, THEN calculate EMI on the approved amount"), you must run them sequentially, not with `gather`.

**Real-world impact:** For local Python functions like ours, the speedup is small (both finish in ms). The gains are huge when tools make **network calls** — e.g., one KB retrieval + one DynamoDB read in parallel = ~200ms instead of ~400ms.

---
# MODULE C — Guardrails + PII Redaction (Compliance)
### Mask Aadhaar/PAN, block toxic content, deny out-of-scope topics

Lending is heavily regulated. Two layers of protection:

1. **PII redaction** — customers paste Aadhaar/PAN numbers; these must never be logged
   or sent to the model in the clear.
2. **Bedrock Guardrails** — block toxic content and topics the agent shouldn't discuss
   (e.g. tax evasion, guaranteed-approval promises).

In [ ]:
# PII redaction — mask Aadhaar, PAN, phone, and email before processing

def redact_pii(text):
    """Redact Indian PII patterns before the text reaches the model or logs."""
    patterns = {
        'AADHAAR': (r'\b\d{4}\s?\d{4}\s?\d{4}\b', '[AADHAAR_REDACTED]'),
        'PAN':     (r'\b[A-Z]{5}\d{4}[A-Z]\b', '[PAN_REDACTED]'),
        'PHONE':   (r'\b[6-9]\d{9}\b', '[PHONE_REDACTED]'),
        'EMAIL':   (r'\b[\w.-]+@[\w.-]+\.\w+\b', '[EMAIL_REDACTED]'),
    }
    found = []
    for label, (pattern, replacement) in patterns.items():
        if re.search(pattern, text):
            found.append(label)
            text = re.sub(pattern, replacement, text)
    return text, found


# Test PII redaction
test_input = ('My PAN is ABCDE1234F, Aadhaar 1234 5678 9012, '
              'phone 9876543210, email rahul@example.com. Am I eligible for a loan?')
clean, found = redact_pii(test_input)
show(test_input, title='Original')
show(clean, title='Redacted')
print(f'✅ PII types redacted: {found}')

### 📖 What this cell does

Implements a **regex-based PII redactor** for the four PII types common in Indian loan applications.

**The four regex patterns:**
| Type | Pattern | Matches |
|------|---------|---------|
| Aadhaar | `\d{4}\s?\d{4}\s?\d{4}` | 12-digit number in any grouping — `123412341234` or `1234 5678 9012` |
| PAN | `[A-Z]{5}\d{4}[A-Z]` | Indian PAN format: `ABCDE1234F` |
| Phone | `[6-9]\d{9}` | Indian mobile numbers (start with 6/7/8/9, 10 digits total) |
| Email | `[\w.-]+@[\w.-]+\.\w+` | Standard email format |

**How it works:** For each PII type, it checks if the pattern matches, records the type in `found`, and replaces the match with a placeholder like `[PAN_REDACTED]`.

**Why redact BEFORE the LLM?**
1. Compliance — Bedrock logs the raw request. If you send Aadhaar, it's now in AWS's logs.
2. Model quality — the LLM doesn't need the actual number to do its job.
3. Defence in depth — even if the LLM prompt leaks somehow, the PII is already masked.

**Limitations:** Regex misses edge cases (e.g., Aadhaar written as `12-34-56-78-90-12`). For production, add a second-pass ML model like Amazon Comprehend PII detection.

In [ ]:
# Wrap the agent so every input is redacted before it reaches the model
def safe_agent_call(agent, user_input):
    """Redact PII, log the safe version, then call the agent."""
    clean_input, pii_found = redact_pii(user_input)
    if pii_found:
        print(f'⚠️  Redacted PII before processing: {pii_found}')
    response = agent(clean_input)
    return str(response)


safe_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor. Use the tools. Never ask for or store PII like Aadhaar or PAN.',
    tools=[check_eligibility, calculate_emi]
)

result = safe_agent_call(
    safe_agent,
    'My PAN is ABCDE1234F. I earn 90000 with score 760. Eligible for 600000?'
)
print('\nAgent response:')
show(result)

### 📖 What this cell does

Wraps every agent call in `safe_agent_call()` — an interceptor that runs PII redaction **before** the request reaches the LLM.

**The pattern:**
```
raw_input → redact_pii → clean_input → agent → response
                ↓
           log which PII types were found (but NOT the values)
```

**Two important defence layers:**
1. **Input redaction** (this function) — masks PII in the *customer's message*
2. **System prompt guardrail** (`"Never ask for or store PII"`) — tells the model not to *echo back* PII it somehow sees

**Notice the system prompt change:** `"Never ask for or store PII like Aadhaar or PAN"`. Even with redaction, we tell the model not to solicit these numbers in the first place. Belt AND suspenders.

**Result in the test:** The PAN `ABCDE1234F` is replaced with `[PAN_REDACTED]` before the LLM ever sees it. The agent still answers the eligibility question correctly using the other data (income, credit score).

In [ ]:
# Bedrock Guardrails — create a guardrail that blocks denied topics and toxic content
bedrock_client = boto3.client('bedrock', region_name=REGION)
GUARDRAIL_ID = None
GUARDRAIL_VERSION = 'DRAFT'

try:
    guardrail = bedrock_client.create_guardrail(
        name='loangenie-guardrail',
        description='Blocks guaranteed-approval promises, tax evasion, and toxic content',
        topicPolicyConfig={
            'topicsConfig': [
                {
                    'name': 'GuaranteedApproval',
                    'definition': 'Promises or guarantees of loan approval without verification',
                    'examples': ['You are guaranteed approval', 'I promise your loan will be approved'],
                    'type': 'DENY'
                },
                {
                    'name': 'TaxEvasion',
                    'definition': 'Advice on evading taxes or hiding income',
                    'examples': ['How do I hide income from the bank', 'How to evade tax on my loan'],
                    'type': 'DENY'
                }
            ]
        },
        contentPolicyConfig={
            'filtersConfig': [
                {'type': 'HATE', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
                {'type': 'INSULTS', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            ]
        },
        blockedInputMessaging='I can only help with legitimate loan eligibility and advisory questions.',
        blockedOutputsMessaging='I cannot provide that information.'
    )
    GUARDRAIL_ID = guardrail['guardrailId']
    print(f'✅ Guardrail created: {GUARDRAIL_ID}')
    ver = bedrock_client.create_guardrail_version(guardrailIdentifier=GUARDRAIL_ID)
    GUARDRAIL_VERSION = ver['version']
    print(f'✅ Guardrail version: {GUARDRAIL_VERSION}')
except bedrock_client.exceptions.ConflictException:
    # Already exists — look it up instead of failing
    existing = bedrock_client.list_guardrails()['guardrails']
    match = next((g for g in existing if g['name'] == 'loangenie-guardrail'), None)
    if match:
        GUARDRAIL_ID = match['id']
        print(f'ℹ️  Guardrail already exists, reusing: {GUARDRAIL_ID}')
except Exception as e:
    print(f'⚠️  Could not create guardrail: {e}')
    print('   If this is AccessDenied, your SageMaker execution role needs bedrock:CreateGuardrail.')
    print('   The rest of the notebook will still run — guarded_call() just won\'t be testable below.')

### 📖 What this cell does

Creates a **Bedrock Guardrail** — AWS's managed content-safety layer that filters unsafe input and output at the model boundary.

**Two policy types configured:**

**1. `topicPolicyConfig` — Denied topics** (business-specific)
- `GuaranteedApproval` — blocks anyone (customer or model) from claiming approval is guaranteed. Critical for regulatory compliance.
- `TaxEvasion` — blocks tax-evasion advice.

Each topic includes:
- `definition` — plain-English description the LLM uses to classify
- `examples` — concrete phrases that should trigger the block
- `type: 'DENY'` — action to take

**2. `contentPolicyConfig` — Generic content filters** (built-in)
- `HATE`, `INSULTS` — set to `HIGH` strength on both input and output

**Idempotency:** The `except ConflictException` block means if you re-run this cell after a guardrail already exists, it fetches and reuses the existing one instead of crashing.

**Custom messages:**
- `blockedInputMessaging` — what the user sees when their input is blocked
- `blockedOutputsMessaging` — what the user sees when the model's output is blocked

**Two versions get created:** A `DRAFT` (editable) and a numbered version (immutable, used in production).

In [ ]:
# Apply the guardrail to a Bedrock call
def guarded_call(prompt, guardrail_id, guardrail_version='DRAFT'):
    """Call Bedrock with a guardrail applied."""
    bedrock_rt = boto3.client('bedrock-runtime', region_name=REGION)
    try:
        resp = bedrock_rt.converse(
            modelId=MODEL_ID,
            messages=[{'role':'user','content':[{'text':prompt}]}],
            guardrailConfig={'guardrailIdentifier':guardrail_id,'guardrailVersion':guardrail_version}
        )
        return resp['output']['message']['content'][0]['text']
    except Exception as e:
        return f'Guardrail response: {e}'

# Test — a denied topic should be blocked
if GUARDRAIL_ID:
    print(guarded_call('Can you guarantee my loan will be approved?', GUARDRAIL_ID, GUARDRAIL_VERSION))
    print('✅ Guarded call pattern ready')
else:
    print('⏭️  Skipped — no guardrail was created above (see warning in the previous cell).')

### 📖 What this cell does

Shows how to **apply the guardrail** to a Bedrock call using the `guardrailConfig` parameter.

**How it's wired:** Just add `guardrailConfig` to any `converse()` or `converse_stream()` call:
```python
resp = bedrock_rt.converse(
    modelId=MODEL_ID,
    messages=[...],
    guardrailConfig={'guardrailIdentifier': GUARDRAIL_ID, 'guardrailVersion': 'DRAFT'}
)
```

That's it — Bedrock runs the guardrail on both input and output automatically. If a denied topic is detected, the response is replaced with your custom `blockedInputMessaging`.

**The test:** *"Can you guarantee my loan will be approved?"* — should trigger the `GuaranteedApproval` deny topic and return: *"I can only help with legitimate loan eligibility and advisory questions."*

**In production:** Wrap **every** LLM call your Lambda makes with the guardrail. It's ~50ms extra latency but adds a critical compliance layer that regulators expect.

---
# MODULE D — Observability
### CloudWatch metrics, request tracing, and per-query cost tracking

You can't run an agent in production without knowing: how many queries, how fast, how
much they cost, and where they fail. This module adds a lightweight observability layer.

In [ ]:
# Per-query cost + latency tracking

# Nova Pro pricing (per 1M tokens) — verify current rates at aws.amazon.com/bedrock/pricing
NOVA_PRO_INPUT_PER_1M  = 0.80
NOVA_PRO_OUTPUT_PER_1M = 3.20

class LoanGenieMetrics:
    """Tracks latency, token usage, and cost per query."""
    def __init__(self):
        self.queries = []

    def track(self, query, response, input_tokens, output_tokens, latency_ms):
        cost = (input_tokens/1e6*NOVA_PRO_INPUT_PER_1M +
                output_tokens/1e6*NOVA_PRO_OUTPUT_PER_1M)
        record = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'query': query[:50],
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'latency_ms': round(latency_ms, 1),
            'cost_usd': round(cost, 6),
        }
        self.queries.append(record)
        return record

    def summary(self):
        if not self.queries: return {'total_queries': 0, 'total_cost_usd': 0, 'avg_latency_ms': 0, 'total_tokens': 0}
        total_cost = sum(q['cost_usd'] for q in self.queries)
        avg_latency = sum(q['latency_ms'] for q in self.queries)/len(self.queries)
        total_tokens = sum(q['input_tokens']+q['output_tokens'] for q in self.queries)
        return {
            'total_queries': len(self.queries),
            'total_cost_usd': round(total_cost, 6),
            'avg_latency_ms': round(avg_latency, 1),
            'total_tokens': total_tokens,
            'projected_cost_per_1000_queries': round(total_cost/len(self.queries)*1000, 2)
        }

metrics = LoanGenieMetrics()
print('✅ Metrics tracker ready')

### 📖 What this cell does

Creates a **`LoanGenieMetrics` class** that tracks four things per query: latency, input tokens, output tokens, and cost.

**Cost math:** Bedrock charges separately for input and output tokens. Nova Pro pricing (as of this lab):
- **Input:** $0.80 per 1M tokens
- **Output:** $3.20 per 1M tokens (4× more expensive — that's why the output is what drives cost)

Formula: `cost = (input_tokens / 1_000_000) × 0.80 + (output_tokens / 1_000_000) × 3.20`

⚠️ **Verify these rates** at aws.amazon.com/bedrock/pricing — they change occasionally.

**The `summary()` method** gives you a session rollup:
- `total_queries` — how many calls in this session
- `total_cost_usd` — cumulative spend
- `avg_latency_ms` — mean response time
- `projected_cost_per_1000_queries` — the killer metric for capacity planning

**Why this matters:** Before adding this, you have no idea what a query costs. After adding this, you can compute *"if we launch to 10,000 users making 3 queries a day, that's ~$X/month"* — critical for pricing decisions.

In [ ]:
# Instrumented agent call — capture tokens, latency, cost via Bedrock converse
bedrock = boto3.client('bedrock-runtime', region_name=REGION)

def tracked_query(prompt):
    """Run a query and capture full observability metrics."""
    start = time.time()
    resp = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{'role':'user','content':[{'text':prompt}]}],
        inferenceConfig={'maxTokens':400,'temperature':0.3}
    )
    latency_ms = (time.time() - start) * 1000

    usage = resp['usage']
    answer = resp['output']['message']['content'][0]['text']
    record = metrics.track(prompt, answer,
        usage['inputTokens'], usage['outputTokens'], latency_ms)

    print(f'📊 {record["latency_ms"]}ms | '
          f'{record["input_tokens"]}+{record["output_tokens"]} tokens | '
          f'${record["cost_usd"]:.6f}')
    return answer

# Run several tracked queries
for q in [
    'What is the minimum credit score for a personal loan?',
    'Explain how EMI is calculated.',
    'What documents does a self-employed applicant need?',
]:
    tracked_query(q)

print('\n=== SESSION SUMMARY ===')
print(json.dumps(metrics.summary(), indent=2))

### 📖 What this cell does

Wraps a Bedrock `converse()` call in **automatic instrumentation** — every call is timed, priced, and recorded.

**Where the token counts come from:** Bedrock returns them in `resp['usage']`:
```python
usage = {'inputTokens': 42, 'outputTokens': 87, 'totalTokens': 129}
```

No estimation needed — these are the actual billed counts from AWS.

**Why `time.time()` before and after:** Simple wall-clock latency measurement in milliseconds. If your agent uses tools, this includes tool execution time too.

**The three test queries** run through in order and each prints its own metrics line:
```
📊 1240ms | 42+87 tokens | $0.000313
```

**The final `metrics.summary()`** aggregates all three into a session-level report — this is what you'd log to CloudWatch or push to a dashboard.

In [ ]:
# Publish custom metrics to CloudWatch
cloudwatch = boto3.client('cloudwatch', region_name=REGION)

def publish_metrics(metrics_obj):
    """Push LoanGenie metrics to CloudWatch for dashboards and alarms."""
    summary = metrics_obj.summary()
    if not summary.get('total_queries'):
        print('No metrics to publish'); return

    try:
        cloudwatch.put_metric_data(
            Namespace='LoanGenie',
            MetricData=[
                {'MetricName':'QueryCount','Value':summary['total_queries'],'Unit':'Count'},
                {'MetricName':'AvgLatency','Value':summary['avg_latency_ms'],'Unit':'Milliseconds'},
                {'MetricName':'TotalCost','Value':summary['total_cost_usd'],'Unit':'None'},
                {'MetricName':'TotalTokens','Value':summary['total_tokens'],'Unit':'Count'},
            ]
        )
        print('✅ Metrics published to CloudWatch namespace: LoanGenie')
        print('   View at: CloudWatch → Metrics → LoanGenie')
    except Exception as e:
        print(f'⚠️  Could not publish to CloudWatch: {e}')
        print('   If AccessDenied, your SageMaker execution role needs cloudwatch:PutMetricData.')

publish_metrics(metrics)

### 📖 What this cell does

Pushes the session's metrics to **CloudWatch** under a custom namespace called `LoanGenie` — so you can build dashboards and alarms in the AWS console.

**The four metrics published:**
| Metric | Unit | Alarm use case |
|--------|------|----------------|
| `QueryCount` | Count | Spike detection / usage tracking |
| `AvgLatency` | Milliseconds | Alarm if p99 latency > 5s |
| `TotalCost` | None (USD) | Alarm if daily spend > $50 |
| `TotalTokens` | Count | Capacity planning |

**Where to view these:** AWS Console → CloudWatch → Metrics → **Custom Namespaces** → `LoanGenie`.

**From there you can:**
- Build a dashboard mixing these with Lambda invocation counts and API Gateway latency
- Set alarms — e.g., SNS notification if `AvgLatency` exceeds 3000ms for 5 minutes
- Export to Grafana / Datadog if you use those

**IAM note:** The SageMaker execution role needs `cloudwatch:PutMetricData`. The lab attached `CloudWatchFullAccess` earlier, which includes it.

In [ ]:
# Simple request tracing — log the full lifecycle of a query

def traced_query(agent, user_query):
    """Trace a query through redaction → agent → response with a trace ID."""
    trace_id = str(uuid.uuid4())[:8]
    trace = {'trace_id': trace_id, 'steps': []}

    def log(step, detail):
        trace['steps'].append({'step': step, 'detail': detail,
                               'ts': datetime.now(timezone.utc).strftime('%H:%M:%S.%f')[:-3]})

    log('received', f'query length {len(user_query)}')
    clean, pii = redact_pii(user_query)
    log('pii_redaction', f'redacted: {pii}' if pii else 'no PII found')

    start = time.time()
    response = str(agent(clean))
    log('agent_response', f'{(time.time()-start)*1000:.0f}ms, {len(response)} chars')

    print(f'🔍 TRACE {trace_id}')
    for s in trace['steps']:
        print(f'   [{s["ts"]}] {s["step"]}: {s["detail"]}')
    return response, trace


demo_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor.',
    tools=[check_eligibility, calculate_emi]
)
response, trace = traced_query(demo_agent,
    'My PAN is ABCDE1234F. Am I eligible for 500000 with income 80000, score 720?')
print(f'\n{response}')

### 📖 What this cell does

Adds **request tracing** — every query gets a unique `trace_id` and each step is timestamped. Essential for debugging when something goes wrong in production.

**The trace structure:**
```
🔍 TRACE a3f4b201
   [12:04:33.421] received: query length 78
   [12:04:33.423] pii_redaction: redacted: ['PAN']
   [12:04:37.891] agent_response: 4468ms, 512 chars
```

**Why 8-char trace IDs?** Short enough to eyeball in logs (`a3f4b201`), long enough to be effectively unique for a single session (`uuid4()[:8]` gives 16^8 = ~4 billion combos).

**In production, you'd:**
1. Log the trace to structured JSON (CloudWatch Logs or S3)
2. Include the `trace_id` in the API response so support can look up any customer's exact request
3. Correlate with Lambda X-Ray traces for end-to-end visibility (Lambda → Bedrock → tools)

**Pattern:** Trace IDs are the "black box flight recorder" of your agent — when a customer complains "the answer was wrong at 3pm yesterday", you need to reconstruct exactly what happened. Trace IDs make that possible.

---
# MODULE E — Deployment to Lambda + API Gateway + Frontend
### Package the agent, deploy it, expose an HTTPS API, and connect the web UI

This is the clean, consolidated deployment path. It replaces the trial-and-error cells
from the working session. Run these in order — each is idempotent (safe to re-run).

**Key fixes baked in:**
- Handler written without `textwrap.dedent` (avoids the leading-indent syntax error)
- IAM role uses the correct `service-role/AWSLambdaBasicExecutionRole` ARN
- Lambda deployed via S3 (Strands package is ~74 MB, over the 69 MB direct limit)
- CORS enabled on API Gateway so the browser frontend can call it

In [ ]:
# [E-1] Create the Lambda IAM role (idempotent)
ROLE_NAME = 'loangenie-lambda-role'
iam = boto3.client('iam', region_name=REGION)

try:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps({
            "Version":"2012-10-17",
            "Statement":[{"Effect":"Allow","Principal":{"Service":"lambda.amazonaws.com"},
                          "Action":"sts:AssumeRole"}]
        })
    )
    print('✅ Role created')
except iam.exceptions.EntityAlreadyExistsException:
    print('✅ Role already exists')
except Exception as e:
    if 'AccessDenied' in str(e):
        print('❌ AccessDenied creating the IAM role.')
        print('   Your SageMaker execution role needs iam:CreateRole, iam:AttachRolePolicy,')
        print('   iam:PutRolePolicy, iam:PassRole — scoped to role/loangenie-* is enough.')
        print('   Ask your AWS admin to attach that, then re-run this cell.')
    raise

for policy in [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
    'arn:aws:iam::aws:policy/AmazonBedrockFullAccess',
    'arn:aws:iam::aws:policy/AmazonDynamoDBFullAccess',
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
]:
    try:
        iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy)
        print(f'✅ {policy.split("/")[-1]}')
    except Exception as e:
        print(f'   {policy.split("/")[-1]}: {e}')

print('Waiting 10s for IAM propagation...')
time.sleep(10)
ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}'
print(f'✅ Role ready: {ROLE_ARN}')

### 📖 What this cell does

Creates the **IAM role that Lambda will assume** at runtime. Without this, Lambda has no permissions and every call fails with `AccessDenied`.

**Two IAM concepts at play:**

**1. Trust policy** (`AssumeRolePolicyDocument`) — WHO can use this role.
Here, only `lambda.amazonaws.com` (the Lambda service) can assume it.

**2. Attached policies** — WHAT this role is allowed to do.
Four managed policies attached:
- `AWSLambdaBasicExecutionRole` — write logs to CloudWatch (required for **every** Lambda)
- `AmazonBedrockFullAccess` — call Bedrock models and KB
- `AmazonDynamoDBFullAccess` — read/write LoanApplicants table
- `AmazonS3FullAccess` — read KB source docs (in production, scope this down)

**Idempotency:** `EntityAlreadyExistsException` is caught, so re-running the cell is safe.

**The 10-second sleep:** IAM changes take a few seconds to propagate across AWS. Without this wait, the next cell might get `Role not found` even though we just created it.

In [ ]:
# [E-2] Write the Lambda handler — NO textwrap (avoids indent error)
HANDLER = """import json
import logging
from strands import Agent, tool
from strands.models import BedrockModel

logger = logging.getLogger()
logger.setLevel(logging.INFO)

@tool
def check_eligibility(monthly_income: float, credit_score: float, loan_amount: float) -> str:
    \"\"\"Check loan eligibility.\"\"\"
    max_loan = monthly_income * 60
    reasons, eligible = [], True
    if credit_score < 650: eligible=False; reasons.append("Credit score below 650")
    if loan_amount > max_loan: eligible=False; reasons.append("Exceeds max eligible")
    if monthly_income < 25000: eligible=False; reasons.append("Income below 25000")
    band = "10.5%-12.5%" if credit_score >= 750 else "12.5%-15%"
    return json.dumps({"eligible":eligible,"max_eligible_amount":int(max_loan),
                       "reasons":reasons or ["Meets all criteria"],"interest_rate_band":band})

@tool
def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    \"\"\"Calculate monthly EMI.\"\"\"
    r = annual_rate/12/100
    emi = principal/tenure_months if r==0 else principal*r*(1+r)**tenure_months/((1+r)**tenure_months-1)
    return json.dumps({"monthly_emi":round(emi,2),"total_payable":round(emi*tenure_months,2),
                       "total_interest":round(emi*tenure_months-principal,2)})

agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-pro-v1:0"),
    system_prompt="You are LoanGenie, a loan eligibility and advisory agent. Use check_eligibility to assess eligibility and calculate_emi for repayments. Format responses in clean markdown with ## headers and **bold** figures. Never promise final approval.",
    tools=[check_eligibility, calculate_emi]
)

def lambda_handler(event, context):
    try:
        body = json.loads(event.get("body", "{}"))
        msg  = body.get("message", "").strip()
        if not msg:
            return {"statusCode":400,"body":json.dumps({"error":"message required"})}
        resp = agent(msg)
        return {
            "statusCode":200,
            "headers":{
                "Content-Type":"application/json",
                "Access-Control-Allow-Origin":"*",
                "Access-Control-Allow-Headers":"Content-Type",
                "Access-Control-Allow-Methods":"POST,OPTIONS"
            },
            "body":json.dumps({"response":str(resp)})
        }
    except Exception as e:
        logger.error(str(e))
        return {"statusCode":500,"body":json.dumps({"error":str(e)})}
"""

with open('/tmp/lambda_handler.py', 'w') as f:
    f.write(HANDLER)

# Verify the file starts at column 0 (no indent on line 2)
with open('/tmp/lambda_handler.py') as f:
    lines = f.readlines()
print('First 2 lines (must have NO leading spaces):')
for i, l in enumerate(lines[:2], 1):
    print(f'  Line {i}: {repr(l)}')
print('✅ Handler written cleanly')

### 📖 What this cell does

Writes the **Lambda function code** to `/tmp/lambda_handler.py` — this will be zipped and deployed in the next cell.

**Why write it as a big string** instead of importing from a `.py` file?
- The notebook lives in SageMaker, but the code needs to run inside Lambda's execution environment.
- Writing it as a triple-quoted string keeps everything self-contained in one notebook.

**⚠️ Critical: No `textwrap.dedent()`**
The original working session used `textwrap.dedent()` which stripped indentation. If the resulting file has any leading whitespace on line 1, Python gives `IndentationError: unexpected indent` — and Lambda fails on cold start. This version writes the string exactly as-is.

**Handler structure:**
1. `import` deps + define the two tools (`check_eligibility`, `calculate_emi`)
2. Create ONE `agent` instance at **module scope** — reused across invocations (warm start optimization)
3. `lambda_handler(event, context)` — the entrypoint AWS calls per request
4. Parses `event['body']` → extracts `message` → calls agent → returns 200 with response

**The CORS headers in the response** (`Access-Control-Allow-Origin: *`) are what let the S3-hosted frontend call this API from the browser. Without them, you get CORS errors.

**The verification at the end** prints the first 2 lines. Both should show `repr(...)` starting with `'import ...'` (no leading spaces).

---
## 🔧 REPLACE THIS — Your S3 Bucket Name

The next cell uses an S3 bucket to stage the Lambda deployment package. It defaults to `loangenie-kb-{ACCOUNT_ID}-us-east-1` (the KB bucket you created in Part 1).

**If your bucket has a different name**, update the `S3_BUCKET` variable in the next cell.

💡 **The cell auto-creates the bucket if it doesn't exist**, so you don't have to create it manually — but the name has to be **globally unique across all of AWS**. Using `{ACCOUNT_ID}` in the name guarantees uniqueness for you.

In [ ]:
# [E-3] Package the agent and upload the zip to S3
S3_BUCKET = f'loangenie-kb-{ACCOUNT_ID}-us-east-1'   # 🔧 Change this if your bucket has a different name
S3_KEY    = 'lambda/loan_agent.zip'
pkg       = '/tmp/loan_pkg'
zip_path  = '/tmp/loan_agent.zip'

s3 = boto3.client('s3', region_name=REGION)

# Verify the bucket exists — create it if it doesn't, instead of failing at upload time
existing_buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
if S3_BUCKET not in existing_buckets:
    print(f'⚠️  Bucket {S3_BUCKET} not found. Creating it...')
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={'LocationConstraint': REGION})
    print(f'✅ Bucket created: {S3_BUCKET}')
else:
    print(f'✅ Bucket exists: {S3_BUCKET}')

print('Installing packages into build dir (2-3 min)...')
shutil.rmtree(pkg, ignore_errors=True); os.makedirs(pkg)
subprocess.run(f'pip install strands-agents strands-agents-tools boto3 -t {pkg} -q',
               shell=True, check=True)
shutil.copy('/tmp/lambda_handler.py', f'{pkg}/lambda_handler.py')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(pkg):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, pkg))
print(f'✅ Packaged: {os.path.getsize(zip_path)/1024/1024:.1f} MB')

s3.upload_file(zip_path, S3_BUCKET, S3_KEY)
print(f'✅ Uploaded: s3://{S3_BUCKET}/{S3_KEY}')

### 📖 What this cell does

**Packages the agent code + all its Python dependencies into a Lambda-compatible zip**, then uploads it to S3.

**Why S3 and not direct upload?** Lambda's direct upload limit is 69 MB. The Strands framework + boto3 + dependencies come to ~74 MB. Any Lambda deployment >69 MB **must** go via S3.

**Step-by-step:**
1. **Check bucket exists** — if not, create it (`us-east-1` uses different `create_bucket` syntax than other regions; the `if REGION` handles both).
2. **`pip install -t /tmp/loan_pkg`** — installs `strands-agents`, `strands-agents-tools`, and `boto3` into a **target directory** (not the system site-packages). This gives us the exact bundle Lambda needs.
3. **Copy `lambda_handler.py`** into the same directory.
4. **Zip everything** with `zipfile.ZIP_DEFLATED` (compressed).
5. **Upload** to `s3://{bucket}/lambda/loan_agent.zip`.

**Takes 2-3 minutes** — mostly the pip install downloading and unpacking packages.

**Output should show `~74 MB packaged`.** If you see much smaller or larger, something's wrong.

In [ ]:
# [E-4] Deploy the Lambda function from S3
LAMBDA_NAME = 'LoanGenieAgent'
lam = boto3.client('lambda', region_name=REGION)

try:
    resp = lam.create_function(
        FunctionName=LAMBDA_NAME, Runtime='python3.12', Role=ROLE_ARN,
        Handler='lambda_handler.lambda_handler',
        Code={'S3Bucket': S3_BUCKET, 'S3Key': S3_KEY},
        MemorySize=1024, Timeout=300,
    )
    print(f'✅ Lambda created: {resp["FunctionArn"]}')
except lam.exceptions.ResourceConflictException:
    resp = lam.update_function_code(
        FunctionName=LAMBDA_NAME, S3Bucket=S3_BUCKET, S3Key=S3_KEY)
    print(f'✅ Lambda updated: {resp["FunctionArn"]}')

LAMBDA_ARN = f'arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}'
print('Waiting 15s for Lambda to be ready...')
time.sleep(15)

# Verify it works before exposing an API
resp = lam.invoke(FunctionName=LAMBDA_NAME, InvocationType='RequestResponse',
    Payload=json.dumps({'body': json.dumps({'message': 'Am I eligible for 500000 with income 80000 and score 720?'})}))
raw = json.loads(resp['Payload'].read())
if raw.get('statusCode') == 200:
    print('✅ Lambda test passed:')
    show(json.loads(raw['body'])['response'])
else:
    print('❌ Lambda error:', raw)

### 📖 What this cell does

**Creates (or updates) the `LoanGenieAgent` Lambda function** from the zip we uploaded, then tests it once before exposing it via API Gateway.

**Key config choices:**
| Setting | Value | Why |
|---------|-------|-----|
| `Runtime` | `python3.12` | Latest supported Python for Lambda |
| `Handler` | `lambda_handler.lambda_handler` | `<filename>.<function_name>` — points to our entry function |
| `MemorySize` | `1024` MB | Enough headroom for Strands + Bedrock SDK. Lower and cold starts get slow |
| `Timeout` | `300` s (5 min) | Long enough for multi-tool agent runs. Default 3s is way too short |
| `Code.S3Bucket/S3Key` | Our uploaded zip | Deploys from S3 (not direct upload) |

**Idempotent:** `ResourceConflictException` (function already exists) → falls back to `update_function_code()`. Safe to re-run.

**The 15-second wait** lets Lambda finish initializing before we invoke it. Without it, you get `ResourceConflictException: The operation cannot be performed at this time.`

**Direct invocation test** — bypasses API Gateway to make sure the Lambda works standalone. If this fails, the API test will fail too. If this passes but the API test fails, the problem is in API Gateway config.

In [ ]:
# [E-5] Create the API Gateway HTTP endpoint with CORS (idempotent — reuses existing API by name)
apigw = boto3.client('apigatewayv2', region_name=REGION)

existing_apis = apigw.get_apis()['Items']
match = next((a for a in existing_apis if a['Name'] == 'LoanGenieAPI'), None)

if match:
    API_ID = match['ApiId']
    print(f'ℹ️  Reusing existing API: {API_ID}')
else:
    api = apigw.create_api(
        Name='LoanGenieAPI', ProtocolType='HTTP',
        CorsConfiguration={
            'AllowOrigins': ['*'],
            'AllowMethods': ['POST', 'OPTIONS'],
            'AllowHeaders': ['Content-Type'],
        }
    )
    API_ID = api['ApiId']
    print(f'✅ API created: {API_ID}')

    integration = apigw.create_integration(
        ApiId=API_ID, IntegrationType='AWS_PROXY',
        IntegrationUri=LAMBDA_ARN, PayloadFormatVersion='2.0')
    apigw.create_route(ApiId=API_ID, RouteKey='POST /advise',
        Target=f'integrations/{integration["IntegrationId"]}')
    apigw.create_stage(ApiId=API_ID, StageName='prod', AutoDeploy=True)

try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME, StatementId='allow-apigw-loangenie',
        Action='lambda:InvokeFunction', Principal='apigateway.amazonaws.com',
        SourceArn=f'arn:aws:execute-api:{REGION}:{ACCOUNT_ID}:{API_ID}/*')
except lam.exceptions.ResourceConflictException:
    pass

API_ENDPOINT = f'https://{API_ID}.execute-api.{REGION}.amazonaws.com/prod/advise'
print('✅ API Gateway deployed!')
print(f'\n📌 PASTE THIS INTO THE FRONTEND CONNECTION BOX:')
print(f'   {API_ENDPOINT}')

### 📖 What this cell does

**Creates an HTTP API in API Gateway** and wires it to the Lambda function. This is how the browser frontend will talk to your agent.

**Four steps happen here:**

**1. Create the API** — `LoanGenieAPI`, HTTP type (v2, cheaper than REST v1).

**2. CORS Configuration** — critical for the S3-hosted frontend to work:
- `AllowOrigins: ['*']` — any origin (in production, restrict to your domain)
- `AllowMethods: ['POST', 'OPTIONS']` — POST for actual calls, OPTIONS for CORS preflight
- `AllowHeaders: ['Content-Type']` — allow JSON content-type header

**3. Create integration + route + stage:**
- **Integration** — the "how" — calls our Lambda via AWS_PROXY (Lambda handles the full request/response format)
- **Route** — `POST /advise` — the URL path
- **Stage** — `prod`, with `AutoDeploy` on so changes go live immediately

**4. Lambda permission** — allow API Gateway to invoke Lambda. Without this, API Gateway gets 500 on every call.

**Final URL format:**  
`https://{api_id}.execute-api.us-east-1.amazonaws.com/prod/advise`

**📌 This URL is what you paste into the frontend's "API Connection" box.**

**Idempotent:** If `LoanGenieAPI` exists, it's reused. Permission conflict is silently swallowed. Safe to re-run.

In [ ]:
# [E-6] Final end-to-end test through the live API
payload = json.dumps({'message':
    'I earn 90000 with credit score 760. Am I eligible for 700000, '
    'and what would the EMI be over 48 months at 11%?'}).encode()
req = urllib.request.Request(API_ENDPOINT, data=payload,
    headers={'Content-Type':'application/json'}, method='POST')

try:
    with urllib.request.urlopen(req) as r:
        data = json.loads(r.read())
    print('✅ Live API response:')
    show(data['response'])
except urllib.error.HTTPError as e:
    print(f'❌ HTTP {e.code}: {e.reason}')
    print(e.read().decode())
    print('\nPulling recent CloudWatch logs to find the real error...')
    logs = boto3.client('logs', region_name=REGION)
    log_group = f'/aws/lambda/{LAMBDA_NAME}'
    try:
        resp = logs.filter_log_events(
            logGroupName=log_group,
            startTime=int((time.time() - 300) * 1000),
            filterPattern='?ERROR ?Exception ?Traceback ?"Task timed out"'
        )
        for ev in resp['events']:
            print(ev['message'])
    except Exception as log_err:
        print(f'   Could not fetch logs: {log_err}')

### 📖 What this cell does

**Full end-to-end test through the live HTTPS API** — same path a real customer would take from the S3 frontend.

**Uses `urllib.request` (stdlib)** so there are no extra dependencies. In production code you'd use `requests` or `httpx`.

**On success:** You'll see the agent's markdown-formatted answer rendered in the notebook. This is the same output the browser frontend will receive.

**On failure — the auto-log-fetching is the star of this cell:**
```python
except urllib.error.HTTPError as e:
    # Fetch recent Lambda CloudWatch logs, filtered for errors
    logs.filter_log_events(
        logGroupName=f'/aws/lambda/{LAMBDA_NAME}',
        startTime=int((time.time() - 300) * 1000),  # last 5 min
        filterPattern='?ERROR ?Exception ?Traceback ?"Task timed out"'
    )
```

Instead of the vague "500 Internal Server Error", it pulls the **actual Python traceback** from CloudWatch Logs. This is huge for debugging.

**Common errors this cell surfaces:**
| Error | Fix |
|-------|-----|
| `pip's dependency resolver...` | Run the dep-fix cell at the top, redeploy |
| `NoSuchBucket` | Check `S3_BUCKET` name in cell E-3 |
| `AccessDenied ... Bedrock` | Attach `AmazonBedrockFullAccess` to `loangenie-lambda-role` |
| `Task timed out` | Bump `Timeout=300` to `Timeout=600` in cell E-4 |

📌 **After this passes, copy the `API_ENDPOINT` and paste it into the frontend's "API Connection" box** (from Section 12 of the lab guide).

---
# 🏆 Capstone — Production LoanGenie
### Every advanced capability combined into one system

Supervisor routing + PII redaction + cost tracking + tracing, all in one call path.

In [ ]:
def production_loangenie(user_query, applicant_id='demo'):
    """Full production pipeline with all advanced capabilities."""
    trace_id = str(uuid.uuid4())[:8]
    start = time.time()

    # 1. PII redaction
    clean_query, pii_found = redact_pii(user_query)

    # 2. Supervisor routing to specialists
    response = str(supervisor(clean_query))

    # 3. Metrics (approximate token counts for demo)
    latency_ms = (time.time() - start) * 1000
    metrics.track(clean_query, response,
                  len(clean_query.split())*2, len(response.split())*2, latency_ms)

    # 4. Structured result
    return {
        'trace_id': trace_id,
        'answer': response,
        'pii_redacted': pii_found,
        'latency_ms': round(latency_ms, 1),
        'applicant_id': applicant_id
    }


# Full end-to-end test
result = production_loangenie(
    'My PAN is ABCDE1234F. I earn 90000 with a credit score of 760. '
    'Am I eligible for 600000, what would the EMI be over 48 months at 12%, '
    'and what documents do I need?',
    applicant_id='applicant-priya-002'
)

print(f'🔍 Trace: {result["trace_id"]}')
print(f'🔒 PII redacted: {result["pii_redacted"]}')
print(f'⏱  Latency: {result["latency_ms"]}ms')
show(result["answer"], title="LoanGenie Response")
print(f'\n📊 Session cost so far: ${metrics.summary()["total_cost_usd"]}')

### 📖 What this cell does

**Combines everything from Modules A-D into a single production-quality function:**

```
User query
    ↓
① PII Redaction (Module C)
    ↓
② Supervisor routing → Specialist agents (Module A)
    ↓
③ Metrics tracking (Module D)
    ↓
④ Structured result: {trace_id, answer, pii_redacted, latency_ms, applicant_id}
```

**Why the structured return?** In a real backend, this dict gets:
- The `answer` returned to the frontend
- The `trace_id` logged for debugging
- The `pii_redacted` field logged for compliance audits
- The `latency_ms` sent to CloudWatch for monitoring
- The `applicant_id` used to persist state in DynamoDB

**The test query hits everything:**
- Contains a PAN (`ABCDE1234F`) → redaction fires
- Multi-part (eligibility + EMI + documents) → all three specialists activated
- Returns single unified markdown response

**Token counting is approximate here** (`len(clean_query.split())*2`) — because Strands doesn't expose raw Bedrock usage from the supervisor call. In your Lambda handler, you'd wire `bedrock.converse()` directly to get exact counts.

**This function is essentially what your Lambda handler should look like in production** — minus the frontend wiring and DynamoDB persistence.

---
# 🧹 Cleanup Advanced Resources
Removes only the resources created in this notebook (guardrail, CloudWatch metrics).

In [ ]:
def cleanup_advanced():
    bedrock_client = boto3.client('bedrock', region_name=REGION)
    # Delete guardrail
    try:
        gids = bedrock_client.list_guardrails()['guardrails']
        for g in gids:
            if g['name'] == 'loangenie-guardrail':
                bedrock_client.delete_guardrail(guardrailIdentifier=g['id'])
                print(f'✅ Deleted guardrail: {g["id"]}')
    except Exception as e:
        print(f'Guardrail: {e}')
    print('✅ CloudWatch metrics expire automatically (no deletion needed)')
    print('\nNote: core resources (KB, DynamoDB, AgentCore) are shared with Part 1 —')
    print('      delete them via Part 1 cleanup when fully done.')

# 🔧 UNCOMMENT to run cleanup:
# cleanup_advanced()
print('👉 Uncomment cleanup_advanced() above to delete the guardrail')

### 📖 What this cell does

**Cleans up ONLY the advanced resources** created in this notebook. Deliberately doesn't touch Part 1 resources (KB, DynamoDB, AgentCore Memory) so you can re-run Part 2 without redoing Part 1.

**What it deletes:**
- **Guardrail** (`loangenie-guardrail`) — actively deleted

**What it doesn't delete (and why):**
- **CloudWatch metrics** — auto-expire after 15 months; no action needed
- **Lambda, API Gateway, IAM role, S3 buckets** — cleaned up via the lab guide's Section 14 cleanup steps (some are shared with Part 1)
- **KB, DynamoDB, AgentCore Memory** — shared with Part 1; delete via the Part 1 cleanup instructions

**Why it's commented out by default:** So running-all-cells doesn't accidentally wipe your resources mid-testing. Uncomment `cleanup_advanced()` when you're truly done.

**For a full teardown**, follow the **Cleanup section** in the lab PDF — it walks through every resource in the right order (empty S3 buckets before deleting them, delete OpenSearch collections, etc.).

---
## ✅ Summary — Advanced LoanGenie

You extended LoanGenie from a working agent into a **production-grade system**:

| Module | Capability | Production value |
|--------|-----------|------------------|
| A | Multi-agent orchestration | Specialist agents, easier to tune & debug |
| B | Streaming + async tools | Real-time UX, lower latency |
| C | Guardrails + PII redaction | Regulatory compliance, data protection |
| D | Observability | Cost control, performance monitoring, tracing |

**What makes this production-ready vs a demo:**
- Every customer input is PII-redacted before it touches the model or logs
- Denied topics (guaranteed approval, tax evasion) are blocked by guardrails
- Every query is cost-tracked and traced — you know exactly what you're spending
- The supervisor pattern means each specialist can be tested and improved independently

---
**Share your build on LinkedIn — tag K21 Academy and Atul Kumar!**
